# NACC data importation, analysis and preparation

Author: Sahana Kowshik

## Libraries and Data Files
- Import necessary libraries, `pandas` and `numpy`.
- Load several CSV files containing data and metadata for analysis.

In [1]:
import pandas as pd
pd.set_option('display.max_columns', None)
import numpy as np
import warnings
warnings.filterwarnings("ignore")

In [2]:
all_nacc = pd.read_csv('/projectnb/vkolagrp/datasets/NACC/csv/processed/investigator_ftldlbd_merged_vnum_unique_nacc65.csv')
# gmm_features = pd.read_csv('/projectnb/vkolagrp/datasets/NACC/csv/processed/gmm_features.csv')
# all_nacc = pd.read_csv('../../data/new_nacc_unique_type_3.csv')
nacc_variable = pd.read_csv('../../data/nacc_variable.csv')
nacc_allowable_code = pd.read_csv('../../data/nacc_allowable_code.csv')
mri = pd.read_csv('/projectnb/vkolagrp/datasets/NACC/csv/raw/investigator_mri_nacc65.csv')

In [3]:
all_nacc = all_nacc.replace({'-4': np.NaN, -4: np.NaN, '-4.0': np.NaN, -4.0: np.NaN})

In [4]:
all_nacc.drop(['PROBAD', 'PROBADIF', 'POSSAD', 'POSSADIF', 'FTLDSUBX', 'OTHBIOM', 'NACCC1', 'NACCC2', 'NEWINF','NACCAGEB', 'WHODIDDX', 'DXMETHOD'], axis=1, inplace=True)

## Data Cleaning

- Fill missing values in variable_id column using forward filling method.

In [5]:
nacc_allowable_code['variable_id_conv'] = nacc_allowable_code['variable_id'].fillna(method='ffill')
nacc_allowable_code.rename(columns={"descriptor": "individual_descriptor"}, inplace=True)

## Data Merging

- Merge `nacc_variable` with `nacc_allowable_code` on variable IDs to consolidate variable descriptions and their corresponding codes.
- Filter merged data for specific UDS forms ('A1', 'A2', 'A3', 'A4', 'A5', 'B1', 'B1a', 'B2','B3', 'B5', 'B6', 'B7', 'B8', 'C1', 'C1, C2', 'C1, C2, C2T', 'C1,C2', 'C2, C2T', 'C2', 'C2T', 'G1', 'D2) to get clinical features.
- Filter merged data for specific UDS forms (B9 and D1) to get diagnosis features.

In [6]:
form = {
    'A1': 'Subject Demographics',
    'A2': 'Co-participant Demographics',
    'A3': 'Subject Family History',
    'A4': 'Subject Medications',
    'A5': 'Subject Health History',
    'B1': 'Physical',
    'B2': 'His and CVD',
    'B3': "Unified Parkinson's Disease Rating Scale (UPDRS)",
    'B4': 'Clinical Dementia Rating (CDR)',
    'B5': 'Neuropsychiatric Inventory Questionnaire (NPI-Q)',
    'B6': 'Geriatric Depression Scale (GDS)',
    'B7': 'Functional Assessment Scale (FAS)',
    'B8': 'Physical/Neurological Exam Findings',
    'B9': 'Clinician Judgment of Symptoms',
    'C': 'Neuropsychological battery Summary Scores',
    'D1': '',
    'D2': 'Medical Conditions',
    'G1': 'Genetic testing',
    'I': 'MRI imaging evidence',
    'P': 'PET imaging evidence'
}

In [7]:
BIOMARKER = ['AMYLPET', 'AMYLCSF', 'FDGAD', 'HIPPATR', 'TAUPETAD', 'CSFTAU', 'FDGFTLD', 'TPETFTLD', 'DATSCAN', 'OTHBIOMX']
MRI = ['CVDIMAG1', 'CVDIMAG2', 'CVDIMAGX', 'IMAGLINF', 'IMAGLAC', 'IMAGMACH', 'IMAGMICH', 'IMAGMWMH', 'IMAGEWMH', 'INFNETW', 'INFWMH', 'MRFTLD']

imag_fea = BIOMARKER + MRI

In [8]:
set(nacc_variable['id']) - set(nacc_allowable_code['variable_id_conv'])

set()

In [9]:
set(nacc_allowable_code['variable_id_conv']) - set(nacc_variable['id'])

set()

In [10]:
merged_df = pd.merge(nacc_variable, nacc_allowable_code, left_on='id', right_on='variable_id_conv', how='outer')
merged_df.loc[merged_df['id'].isin(MRI), 'form_id'] = 'I'
merged_df.loc[merged_df['id'].isin(BIOMARKER), 'form_id'] = 'P'
form_id_desc = []
for i in list(merged_df['form_id']):
    try:
        form_id_desc.append(form[i])
    except:
        form_id_desc.append(np.NaN)
merged_df['form_id_desc'] = form_id_desc
merged_df = merged_df.sort_values('form_id')

forms = ['A1', 'A2', 'A3', 'A4', 'A5', 'B1', 'B1a', 'B2','B3', 'B5', 'B6', 'B7', 'B8', 'C', 'G1', 'D2']
merged_df_patient = merged_df[(merged_df['form_id'].isin(forms)) | (merged_df['variable_id_conv'].isin(imag_fea))]
merged_df_diag = merged_df[(merged_df['form_id'].isin(['D1'])) & (~merged_df['variable_id_conv'].isin(imag_fea))]


## Data Transformation

- Generate dictionaries for 
    - variable name and text diagnosis
    - descriptor codes and text descriptions for each code.

In [11]:
diag_dict = dict(zip(merged_df_diag['id'], zip(merged_df_diag['form_id_desc'], merged_df_diag['descriptor'])))
# diag_dict = dict(zip(merged_df_diag['id'], merged_df_diag['descriptor']))
code_dict = {
    id_value: {
        (row['code_1'] if pd.isna(row['code_2']) else "range"):
        (row['individual_descriptor'] if pd.isna(row['code_2']) else (int(row['code_1']), int(row['code_2'])))
        for index, row in group.iterrows()
    }
    for id_value, group in merged_df_diag.groupby('id')
}

In [12]:
patient_dict = dict(zip(merged_df_patient['id'], zip(merged_df_patient['form_id_desc'], merged_df_patient['descriptor'])))
# patient_dict = dict(zip(merged_df_patient['id'], merged_df_patient['descriptor']))
patient_code_dict = {
    id_value: {
        (row['code_1'] if pd.isna(row['code_2']) else "range"):
        (row['individual_descriptor'] if pd.isna(row['code_2']) else (int(row['code_1']), int(row['code_2'])))
        for index, row in group.iterrows()
    }
    for id_value, group in merged_df_patient.groupby('id')
}

In [13]:
patient_dict['AMYLPET']

('PET imaging evidence', 'Abnormally elevated amyloid on PET')

In [14]:
patient_code_dict['AMYLPET']

{'0': 'Absent',
 '1': 'Present',
 '8': 'Unknown/not assessed',
 '-4': 'Not available: UDS form submitted did not collect data in this way, or a skip pattern precludes response to this question'}

## Patient Record Dictionary Creation

- Filter the record columns for NACC data.
- Iterate through filtered data to map NACC IDs to corresponding record texts.

In [15]:
patient_dict.update({f"DRUG{i}" : patient_dict["DRUG1-DRUG40"] for i in range(1,41)})
patient_dict = {k: v for k, v in patient_dict.items() if k in all_nacc.columns}
filtered_nacc = all_nacc[list(patient_dict.keys()) + ["NACCID"]].replace({-4: np.NaN, "-4": np.NaN})
remove_cols = set(filtered_nacc.columns).intersection(set(['PROBAD', 'PROBADIF', 'POSSAD', 'POSSADIF', 'FTLDSUBX', 'OTHBIOM', 'NACCC1', 'NEWINF','NACCAGEB']))
filtered_nacc.drop(remove_cols, inplace=True, axis=1)
remove_cols

set()

In [16]:
import pandas as pd
import json
from tqdm import tqdm

# Assuming you have the necessary data and dictionaries: patient_dict, patient_code_dict
# Ensure the folder for storing JSON files exists or is created beforehand
patient_texts = []
for i, row in tqdm(filtered_nacc.iterrows(), total=len(filtered_nacc)):
    patient_data = {}
    for k, v in dict(row).items():
        if k not in row or k == "NACCID":
            continue
        
        # if (k in patient_code_dict) and ("range" in patient_code_dict[k]) and ("Demographics" not in patient_dict[k][0]):
        #     val = patient_dict[k][0] + patient_dict[k][1] + " (range " + str(patient_code_dict[k]["range"]).replace(',', ' -') + ')'
        # else:
        #     val = patient_dict[k][0] + patient_dict[k][1]
        
        if patient_dict[k][0] not in patient_data:
            patient_data[patient_dict[k][0]] = {}
            
        if (k in patient_code_dict) and ("range" in patient_code_dict[k]) and ("Demographics" not in patient_dict[k][0]):
            val = patient_dict[k][1] + " (range " + str(patient_code_dict[k]["range"]).replace(',', ' -') + ')'
        else:
            val = patient_dict[k][1]
        
        if not pd.isna(v):
            if not isinstance(v, str):
                if str(int(v)) in patient_code_dict[k]:
                    try:
                        if ('unknown' in patient_code_dict[k][str(int(v))].lower()) or ('not applicable' in patient_code_dict[k][str(int(v))].lower()) or ('not available' in patient_code_dict[k][str(int(v))].lower()) or ('did not report' in patient_code_dict[k][str(int(v))].lower()):
                            continue
                    except:
                        print(k, v, patient_code_dict[k][str(int(v))])
                    patient_data[patient_dict[k][0]][val.replace(';', ',')] = patient_code_dict[k][str(int(v))].replace(';', ',')
                elif "range" in patient_code_dict[k] and int(v) in range(patient_code_dict[k]["range"][0], patient_code_dict[k]["range"][1]+1):
                    if isinstance(v, str) and (('unknown' in v.lower()) or ('not applicable' in v.lower()) or ('not available' in v.lower()) or ('did not report' in v.lower())):
                        continue
                    patient_data[patient_dict[k][0]][val.replace(';', ',')] = v
            else:
                patient_data[patient_dict[k][0]][val.replace(';', ',')] = v
                
    patient_data_str = "```json\n{\n"
    for key, value in patient_data.items():
        if len(value) == 0:
            continue
        patient_data_str += f'\t"{key}"' + ': {\n'
        for k, v in value.items():
            if isinstance(v, str):
                patient_data_str += f'\t\t"{k}": "{v}",\n'
            else:
                patient_data_str += f'\t\t"{k}": {v},\n'
        patient_data_str += '\t},\n'
    patient_data_str += '}\n```'
    patient_texts.append(patient_data_str)
    # # Write to a JSON file
    # with open(f'patient_data_{i}.json', 'w') as f:
    #     json.dump(patient_data, f, ensure_ascii=False, indent=4)
    
    # break


100%|██████████| 188700/188700 [04:33<00:00, 690.72it/s]


In [17]:
print(patient_texts[0])

```json
{
	"Subject Demographics": {
		"Living situation": "Lives with spouse or partner",
		"Primary language": "English",
		"Level of independence": "Able to live independently",
		"Race": "White",
		"Second race": "None reported",
		"Hispanic/Latino ethnicity": "No",
		"Primary reason for coming to ADC": "To participate in a research study",
		"Principal referral source": "Other",
		"Is the subject left- or right-handed?": "Right-handed",
		"Type of residence": "Single- or multi-family private residence (apartment, condo, house)",
		"Subject's age at visit": 62,
		"Marital status": "Married",
		"Years of education": 16,
		"Subject's sex": "Female",
		"Subject's month of birth": 2,
		"Subject's year of birth": 1944,
		"Third race": "None reported",
		"Derived NIH race definitions": "White",
	},
	"Co-participant Demographics": {
		"Co-participant's relationship to subject": "Friend, neighbor, or someone known through family, friends, work, or community",
		"Derived NIH race definition

In [18]:
# from tqdm import tqdm

# patient_texts = []
# for i, row, in tqdm(filtered_nacc.iterrows()):
#     st = ""
#     for k, v in dict(row).items():
#         if k not in row:
#             continue
#         if k == "NACCID":
#             continue
        
#         if (k in patient_code_dict) and ("range" in patient_code_dict[k]) and ("Demographics" not in patient_dict[k][0]):
#             # print(patient_dict[k][0] + patient_dict[k][1] + " (range " + str(patient_code_dict[k]["range"]).replace(',', ' -') + ')')
#             # continue
#             val = patient_dict[k][0] + patient_dict[k][1] + " (range " + str(patient_code_dict[k]["range"]).replace(',', ' -') + ')'
#         else:
#             val = patient_dict[k][0] + patient_dict[k][1]
        
#         if not pd.isna(v):
#             if not isinstance(v, str):
#                 if str(int(v)) in patient_code_dict[k]:
#                     try:
#                         if ('unknown' in patient_code_dict[k][str(int(v))].lower()) | ('not applicable' in patient_code_dict[k][str(int(v))].lower()) | ('not available' in patient_code_dict[k][str(int(v))].lower()) | ('did not report' in patient_code_dict[k][str(int(v))].lower()):
#                             continue
#                     except:
#                         print(k, v, patient_code_dict[k][str(int(v))])
#                     st += f"{val.replace(';', ',')}: {patient_code_dict[k][str(int(v))].replace(';', ',')}; "
#                 elif "range" in patient_code_dict[k] and int(v) in range(patient_code_dict[k]["range"][0], patient_code_dict[k]["range"][1]+1):
#                     if isinstance(v, str) and (('unknown' in v.lower()) | ('not applicable' in v.lower()) | ('not available' in v.lower()) | ('did not report' in v.lower())):
#                         continue
#                     st += f"{val.replace(';', ',')}: {v}; "
#             else:
#                 st += f"{val.replace(';', ',')}: {v}; "
        
#     patient_texts.append(st)
#     # break

In [19]:
len(patient_texts)

188700

## Diagnosis Dictionary Creation

- Filter the diagnostic columns for NACC data.
- Iterate through filtered data to map NACC IDs to corresponding diagnostic texts.

In [20]:
from tqdm import tqdm
avail_keys = set(all_nacc.columns).intersection(set(list(diag_dict.keys()) + ['NACCID']))
filtered_diag_nacc = all_nacc[list(avail_keys)].replace({-4: np.NaN, "-4": np.NaN})
remove_cols = set(filtered_diag_nacc.columns).intersection(set(['PROBAD', 'PROBADIF', 'POSSAD', 'POSSADIF', 'FTLDSUBX', 'OTHBIOM'] + imag_fea))
filtered_diag_nacc.drop(remove_cols, inplace=True, axis=1)

In [21]:
remove_cols

set()

In [22]:
import pandas as pd
import json
from tqdm import tqdm

# Make sure to define or import your diag_dict and code_dict appropriately
# Ensure the folder for storing JSON files exists or is created beforehand
diag_texts = []
for i, row in tqdm(filtered_diag_nacc.iterrows(), total=len(filtered_diag_nacc)):
    diag_data = {}
    for k, v in dict(row).items():
        if k == "NACCID":
            continue
        
        val = diag_dict[k][1]# + diag_dict[k][1]
        
        if not pd.isna(v):
            if not isinstance(v, str):
                if str(int(v)) in code_dict[k]:
                    try:
                        if ('unknown' in code_dict[k][str(int(v))].lower()) or ('not applicable' in code_dict[k][str(int(v))].lower()) or ('not available' in code_dict[k][str(int(v))].lower()) or ('did not report' in code_dict[k][str(int(v))].lower()):
                            continue
                    except:
                        print(k, v, code_dict[k][str(int(v))])
                    diag_data[val.replace(';', ',')] = code_dict[k][str(int(v))].replace(';', ',')
                elif "range" in code_dict[k] and int(v) in range(code_dict[k]["range"][0], code_dict[k]["range"][1]+1):
                    if isinstance(v, str) and (('unknown' in v.lower()) or ('not applicable' in v.lower()) or ('not available' in v.lower()) or ('did not report' in v.lower())):
                        continue
                    diag_data[val.replace(';', ',')] = v
            else:
                diag_data[val.replace(';', ',')] = v
    
    diag_data_str = "```json\n{\n"
    for key, value in diag_data.items():
        if isinstance(value, str):
            v = value.replace(' by clinician', '').replace(" (D1 form)", '')
            diag_data_str += f'\t"{key}": "{v}",\n'
        else:
            diag_data_str += f'\t"{key}": {value},\n'
        
    diag_data_str += '}\n```'
    diag_texts.append(diag_data_str)
    # break


100%|██████████| 188700/188700 [00:55<00:00, 3404.75it/s]


In [23]:
# diag_texts = []
# for i, row, in tqdm(filtered_diag_nacc.iterrows()):
#     st = ""
#     for k, v in dict(row).items():
#         if k == "NACCID":
#             continue
#         val = diag_dict[k][0] + diag_dict[k][1]
                
#         if not pd.isna(v):
#             if not isinstance(v, str):
#                 if str(int(v)) in code_dict[k]:
#                     try:
#                         if ('unknown' in code_dict[k][str(int(v))].lower()) | ('not applicable' in code_dict[k][str(int(v))].lower()) | ('not available' in code_dict[k][str(int(v))].lower()) | ('did not report' in code_dict[k][str(int(v))].lower()):
#                             continue
#                     except:
#                         print(k, v, code_dict[k][str(int(v))])
#                     st += f"{val.replace(';', ',')}: {code_dict[k][str(int(v))].replace(';', ',')}; "
#                 elif "range" in code_dict[k] and int(v) in range(code_dict[k]["range"][0], code_dict[k]["range"][1]+1):
#                     if isinstance(v, str) and (('unknown' in v.lower()) | ('not applicable' in v.lower()) | ('not available' in v.lower()) | ('did not report' in v.lower())):
#                         continue
#                     st += f"{val.replace(';', ',')}: {v}; "
#             else:
#                 st += f"{val.replace(';', ',')}: {v}; "
        
#     diag_texts.append(st)

In [24]:
print(diag_texts[0])

```json
{
	"Presumptive etiologic diagnosis of the cognitive disorder - Other 1 (specify)": "No",
	"Presumptive etiologic diagnosis of the cognitive disorder - Stroke": "No",
	"Primary, contributing, or non-contributing cause of cognitive impairment - CNS neoplasm": "Cognitively impaired but no CNS neoplasm diagnosis",
	"Primary, contributing, or non-contributing cause of cognitive impairment - systemic disease/ medical illness": "Cognitively impaired but no diagnosis of impairment due to systemic disease/medical illness",
	"Primary, contributing, or non-contributing cause of cognitive impairment - behavioral frontotemporal dementia (bvFTD)": "Cognitively impaired but not bvFTD diagnosis",
	"Presumptive etiologic diagnosis of the cognitive disorder - Normal-pressure hydrocephalus (NPH)": "No (assumed assessed and found not present)",
	"Primary progressive aphasia (PPA) with cognitive impairment": "No",
	"Presumptive etiologic diagnosis of the cognitive disorder - Undetermined etiology"

In [25]:
len(diag_texts)

188700

## Data Export

- Save the diagnosis dictionary to a JSON file for further use.

In [26]:
all_nacc["diag_summary"] = diag_texts
all_nacc["patient_summary"] = patient_texts

In [27]:
import os
directory_path = "../../data/1022/csv_to_txt"
if not os.path.exists(directory_path):
    os.makedirs(directory_path, exist_ok=True)

all_nacc.to_csv(f"{directory_path}/all_nacc_csv_to_txt.csv", index=False)